In [37]:
# ── Cell 1 — Dependencies ─────────────────────────────────────────────────
import importlib, subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=False)

_pip('python-chess')
_pip('tqdm')
_pip('wandb')
# _pip('mlflow')  # Uncomment if AZ_CONFIG['tracking_backend'] = 'mlflow'

print('✅ Install complete')


✅ Install complete


In [38]:
# ── Cell 2 — Detect device + T4×2 guard ─────────────────────────────────
# FIX: Thêm guard chỉ chạy trên T4×2.
# FIX: Bỏ torch.backends.cuda.enable_flash_sdp — P100 là compute capability 6.0,
#      Flash Attention yêu cầu >= 7.0 (Turing/Volta), gọi flag đó trên P100 gây lỗi
#      RuntimeError: No kernel for this device.
import torch

assert torch.cuda.is_available(), 'Cần GPU!'

n_gpus = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)

# Guard: chỉ cho phép T4 (dùng bộ nhớ 16GB × 2 để tính batch)
assert 'T4' in gpu_name, (
    f'Notebook này chỉ chạy trên Tesla T4. '
    f'GPU hiện tại: {gpu_name}. '
    f'Trên P100 dùng batch_size nhỏ hơn và bỏ flash sdp.'
)

DEVICE = torch.device('cuda')
print(f'✅ GPU: {n_gpus}× {gpu_name}')
print(f'🖥️  DEVICE = {DEVICE}')

# FIX: Version-safe AMP helpers.
# torch.amp.GradScaler('cuda') và torch.amp.autocast('cuda') được thêm vào
# PyTorch 2.0. Trên môi trường Kaggle cũ hơn (PyTorch 1.x, thường đi kèm P100)
# cần dùng torch.cuda.amp.* API — available từ PyTorch 1.6.
_NEW_AMP = hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler')
print(f'AMP API: {"torch.amp (2.x)" if _NEW_AMP else "torch.cuda.amp (1.x compat)"}')

def make_grad_scaler(enabled=True):
    """Tạo GradScaler tương thích cả PyTorch 1.x và 2.x."""
    if _NEW_AMP:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)

from contextlib import contextmanager

@contextmanager
def amp_autocast(enabled=True):
    """Context manager AMP tương thích cả PyTorch 1.x và 2.x."""
    if _NEW_AMP:
        with torch.amp.autocast('cuda', enabled=enabled):
            yield
    else:
        with torch.cuda.amp.autocast(enabled=enabled):
            yield


✅ GPU: 2× Tesla T4
🖥️  DEVICE = cuda
AMP API: torch.amp (2.x)


In [ ]:
# ── Cell 3 — Config ───────────────────────────────────────────────────────
import os, glob, math

PGN_SOURCE = '/kaggle/input/datasets/solirus/pgn-files'
SAVE_DIR   = '/kaggle/working/az_chess_transformer'

for d in [SAVE_DIR, f'{SAVE_DIR}/checkpoints', f'{SAVE_DIR}/models']:
    os.makedirs(d, exist_ok=True)

VAL_SPLIT       = 0.05
_all_pgn        = sorted(glob.glob(os.path.join(PGN_SOURCE, '*.pgn')))
_n_val          = max(1, math.ceil(len(_all_pgn) * VAL_SPLIT))
TRAIN_PGN_FILES = _all_pgn[_n_val:]
VAL_PGN_FILES   = _all_pgn[:_n_val]

print(f'Train: {len(TRAIN_PGN_FILES)} files | Val: {len(VAL_PGN_FILES)} files')

AZ_CONFIG = {
    'input_planes':      119,
    'action_size':       4672,

    # Transformer
    'd_model':           256,
    'n_heads':           8,
    'n_layers':          12,
    'ffn_dim':           1024,
    'dropout':           0.1,

    # Training — T4×2: batch 256 total (128 per GPU với DataParallel)
    'batch_size':        256,
    'learning_rate':     1e-4,
    'warmup_steps':      2_000,
    'lr_decay_steps':    200_000,
    'weight_decay':      1e-4,
    'epochs':            5,
    'num_workers':       4,

    # Data filter — RELAXED to ensure we have training data
    # If you get empty dataset, try: rating_filter=0, min_moves=1, skip_bullet=False
    'rating_filter':     1600,    # Reduced from 1800 for more data
    'min_moves':         5,       # Reduced from 15 (accept shorter games)
    'skip_bullet':       False,   # Allow bullet games

    'save_dir':          SAVE_DIR,
    'save_every_epoch':  True,

    # Tracking: 'wandb', 'mlflow', or 'none'
    # Kaggle W&B online: add WANDB_API_KEY in Kaggle Secrets. Without it, W&B runs offline.
    'tracking_backend':  'wandb',
    'wandb_project':     'az-chess-transformer',
    'wandb_entity':      None,
    'wandb_mode':        'online',
    'mlflow_uri':        f'file:{SAVE_DIR}/mlruns',
    'mlflow_experiment': 'az-chess-transformer',
    'tracking_upload_checkpoints': False,
}

print('\n⚙️  Config:')
for k, v in AZ_CONFIG.items():
    print(f'   {k:<22} = {v}')

print('\n💡 If dataset is still empty, try:')
print('   AZ_CONFIG["rating_filter"] = 0')
print('   AZ_CONFIG["min_moves"] = 1')

Train: 0 files | Val: 0 files

⚙️  Config:
   input_planes           = 119
   action_size            = 4672
   d_model                = 256
   n_heads                = 8
   n_layers               = 12
   ffn_dim                = 1024
   dropout                = 0.1
   batch_size             = 256
   learning_rate          = 0.0001
   warmup_steps           = 2000
   lr_decay_steps         = 200000
   weight_decay           = 0.0001
   epochs                 = 5
   num_workers            = 4
   rating_filter          = 1800
   min_moves              = 15
   skip_bullet            = True
   save_dir               = /kaggle/working/az_chess_transformer
   save_every_epoch       = True


In [ ]:
# ── Cell 4 — Optional experiment tracking ───────────────────────────────────
# Supports W&B on Kaggle and local-file MLflow. Training still works if tracking fails.
import os

TRACKER = {'backend': 'none', 'run': None}

def _flatten_for_tracking(d, prefix=''):
    out = {}
    for k, v in d.items():
        key = f'{prefix}{k}'
        if isinstance(v, dict):
            out.update(_flatten_for_tracking(v, prefix=f'{key}.'))
        elif isinstance(v, (str, int, float, bool)) or v is None:
            out[key] = v
    return out

def _try_wandb_login():
    api_key = os.environ.get('WANDB_API_KEY')
    if api_key is None:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception:
            api_key = None
    if api_key:
        import wandb
        wandb.login(key=api_key, relogin=True)
        return True
    return False

def setup_tracking(cfg, extra=None):
    backend = str(cfg.get('tracking_backend', 'none')).lower()
    if backend == 'none':
        print('Tracking disabled')
        return TRACKER

    payload = dict(cfg)
    if extra:
        payload.update(extra)

    try:
        if backend == 'wandb':
            import wandb
            has_key = _try_wandb_login()
            requested_mode = cfg.get('wandb_mode', 'online')
            mode = requested_mode if has_key or requested_mode != 'online' else 'offline'
            run = wandb.init(
                project=cfg.get('wandb_project', 'az-chess-transformer'),
                entity=cfg.get('wandb_entity'),
                config=payload,
                mode=mode,
                name=cfg.get('run_name'),
                reinit=True,
            )
            TRACKER.update({'backend': 'wandb', 'run': run})
            print(f'Tracking: W&B ({mode})')
        elif backend == 'mlflow':
            import mlflow
            mlflow.set_tracking_uri(cfg.get('mlflow_uri', f'file:{SAVE_DIR}/mlruns'))
            mlflow.set_experiment(cfg.get('mlflow_experiment', 'az-chess-transformer'))
            run = mlflow.start_run(run_name=cfg.get('run_name'))
            mlflow.log_params(_flatten_for_tracking(payload))
            TRACKER.update({'backend': 'mlflow', 'run': run})
            print(f'Tracking: MLflow ({mlflow.get_tracking_uri()})')
        else:
            print(f'Unknown tracking_backend={backend!r}; tracking disabled')
    except Exception as e:
        print(f'Tracking disabled after init failure: {e}')
        TRACKER.update({'backend': 'none', 'run': None})
    return TRACKER

def log_tracking(metrics, step=None):
    backend = TRACKER.get('backend')
    if backend == 'wandb':
        import wandb
        wandb.log(metrics, step=step)
    elif backend == 'mlflow':
        import mlflow
        clean = {k: float(v) for k, v in metrics.items() if isinstance(v, (int, float))}
        mlflow.log_metrics(clean, step=step)

def log_tracking_artifact(path, name=None):
    if not path or not os.path.exists(path):
        return
    backend = TRACKER.get('backend')
    if backend == 'wandb':
        import wandb
        wandb.save(path, base_path=os.path.dirname(path))
    elif backend == 'mlflow':
        import mlflow
        mlflow.log_artifact(path, artifact_path=name)

def finish_tracking():
    backend = TRACKER.get('backend')
    if backend == 'wandb':
        import wandb
        wandb.finish()
    elif backend == 'mlflow':
        import mlflow
        mlflow.end_run()
    TRACKER.update({'backend': 'none', 'run': None})

print('✅ Tracking helpers ready')


In [40]:
# ── Cell 5 — Board encoder — UNCHANGED ───────────────────────────────────
import chess
import numpy as np

PIECE_TO_PLANE = {
    (chess.PAWN,   chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK,   chess.WHITE): 3,
    (chess.QUEEN,  chess.WHITE): 4,
    (chess.KING,   chess.WHITE): 5,
    (chess.PAWN,   chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK,   chess.BLACK): 9,
    (chess.QUEEN,  chess.BLACK): 10,
    (chess.KING,   chess.BLACK): 11,
}

def _encode_board_planes(board: chess.Board, flip: bool) -> np.ndarray:
    planes = np.zeros((8, 8, 14), dtype=np.float32)
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece is None: continue
        rank, file = divmod(sq, 8)
        if flip:
            rank  = 7 - rank
            color = not piece.color
        else:
            color = piece.color
        planes[rank, file, PIECE_TO_PLANE[(piece.piece_type, color)]] = 1.0
    return planes

def board_history_to_planes(boards: list, current_turn: chess.Color) -> np.ndarray:
    flip    = (current_turn == chess.BLACK)
    history = np.zeros((8, 8, 112), dtype=np.float32)
    for i, b in enumerate(boards[:8]):
        enc = _encode_board_planes(b, flip)
        history[:, :, i*14:(i+1)*14] = enc

    board = boards[0]
    meta  = np.zeros((8, 8, 7), dtype=np.float32)
    meta[:, :, 0] = 1.0
    if not flip:
        meta[:, :, 1] = float(board.has_kingside_castling_rights(chess.WHITE))
        meta[:, :, 2] = float(board.has_queenside_castling_rights(chess.WHITE))
        meta[:, :, 3] = float(board.has_kingside_castling_rights(chess.BLACK))
        meta[:, :, 4] = float(board.has_queenside_castling_rights(chess.BLACK))
    else:
        meta[:, :, 1] = float(board.has_kingside_castling_rights(chess.BLACK))
        meta[:, :, 2] = float(board.has_queenside_castling_rights(chess.BLACK))
        meta[:, :, 3] = float(board.has_kingside_castling_rights(chess.WHITE))
        meta[:, :, 4] = float(board.has_queenside_castling_rights(chess.WHITE))
    meta[:, :, 5] = board.halfmove_clock / 100.0
    meta[:, :, 6] = board.fullmove_number / 100.0

    planes = np.concatenate([history, meta], axis=-1)
    return planes.transpose(2, 0, 1).astype(np.float32)

print('✅ Board encoder ready')


✅ Board encoder ready


In [41]:
# ── Cell 6 — Move encoder ─────────────────────────────────────────────────
# BUG FIX: Phiên bản cũ dùng `hash(move.uci()) % ACTION_SIZE` — hoàn toàn sai.
# hash() trả về giá trị ngẫu nhiên, khiến policy target ánh xạ vào bucket ngẫu
# nhiên không liên quan đến nước đi thực sự. Model học noise thay vì cờ.
# FIX: Khôi phục AlphaZero encoding chuẩn: action_idx = from_sq * 73 + move_type
import chess
import numpy as np

_QUEEN_DIRS        = [(0,1),(0,-1),(1,0),(-1,0),(1,1),(1,-1),(-1,1),(-1,-1)]
_KNIGHT_DELTAS     = [(2,1),(2,-1),(-2,1),(-2,-1),(1,2),(1,-2),(-1,2),(-1,-2)]
_UNDERPROMO_PIECES = [chess.KNIGHT, chess.BISHOP, chess.ROOK]
_UNDERPROMO_DIRS   = [-1, 0, 1]

def _build_move_index():
    idx = {}
    # Queen-like (0–55): 8 hướng × 7 khoảng cách
    for dir_i, (dr, df) in enumerate(_QUEEN_DIRS):
        for dist in range(1, 8):
            move_type = dir_i * 7 + (dist - 1)
            for from_sq in chess.SQUARES:
                r0, f0 = chess.square_rank(from_sq), chess.square_file(from_sq)
                r1, f1 = r0 + dr*dist, f0 + df*dist
                if 0 <= r1 < 8 and 0 <= f1 < 8:
                    idx[(from_sq, chess.square(f1, r1), None)] = from_sq*73 + move_type
    # Knight (56–63)
    for k_i, (dr, df) in enumerate(_KNIGHT_DELTAS):
        move_type = 56 + k_i
        for from_sq in chess.SQUARES:
            r0, f0 = chess.square_rank(from_sq), chess.square_file(from_sq)
            r1, f1 = r0+dr, f0+df
            if 0 <= r1 < 8 and 0 <= f1 < 8:
                idx[(from_sq, chess.square(f1, r1), None)] = from_sq*73 + move_type
    # Underpromotion (64–72)
    for p_i, piece in enumerate(_UNDERPROMO_PIECES):
        for d_i, df in enumerate(_UNDERPROMO_DIRS):
            move_type = 64 + p_i*3 + d_i
            for from_sq in chess.SQUARES:
                r0, f0 = chess.square_rank(from_sq), chess.square_file(from_sq)
                if r0 != 6: continue
                f1 = f0 + df
                if 0 <= f1 < 8:
                    idx[(from_sq, chess.square(f1, 7), piece)] = from_sq*73 + move_type
    return idx

MOVE_TO_IDX = _build_move_index()
ACTION_SIZE  = 64 * 73  # 4672

def move_to_idx(move: chess.Move, flip: bool = False):
    promo   = move.promotion
    if promo == chess.QUEEN: promo = None
    from_sq = move.from_square
    to_sq   = move.to_square
    if flip:
        from_sq = chess.square_mirror(from_sq)
        to_sq   = chess.square_mirror(to_sq)
    return MOVE_TO_IDX.get((from_sq, to_sq, promo))

print(f'✅ Move encoder ready — {len(MOVE_TO_IDX):,} moves indexed (ACTION_SIZE={ACTION_SIZE})')


✅ Move encoder ready — 1,858 moves indexed (ACTION_SIZE=4672)


In [42]:
# ── Cell 7 — PGN filter ──────────────────────────────────────────────────
# BUG FIX: Phiên bản cũ thiếu kiểm tra min_moves, skip_bullet và result filter.
# Game kết thúc '*' (chưa xong) sẽ được đưa vào training với value=0 (draw),
# làm nhiễu value head.
import chess

def build_value_label(result_str: str, turn: chess.Color) -> float:
    if result_str == '1-0':
        return 1.0 if turn == chess.WHITE else -1.0
    elif result_str == '0-1':
        return 1.0 if turn == chess.BLACK else -1.0
    return 0.0  # draw

def passes_pgn_filter(game, cfg: dict) -> bool:
    headers = game.headers

    # FIX: Lọc game chưa kết thúc
    result = headers.get('Result', '*')
    if result not in ('1-0', '0-1', '1/2-1/2'):
        return False

    # Rating filter
    min_elo = cfg.get('rating_filter', 0)
    if min_elo > 0:
        try:
            w = int(headers.get('WhiteElo', '0') or '0')
            b = int(headers.get('BlackElo', '0') or '0')
            if w < min_elo or b < min_elo:
                return False
        except ValueError:
            return False

    # FIX: Bỏ bullet (< 3 phút)
    if cfg.get('skip_bullet', False):
        tc = headers.get('TimeControl', '')
        if tc and tc != '-':
            try:
                if int(tc.split('+')[0]) < 180:
                    return False
            except ValueError:
                pass

    # FIX: Lọc game quá ngắn
    min_moves = cfg.get('min_moves', 0)
    if min_moves > 0:
        if sum(1 for _ in game.mainline_moves()) < min_moves:
            return False

    return True

print('✅ PGN filter ready')


✅ PGN filter ready


In [49]:
# ── Cell 8 — Dataset ──────────────────────────────────────────────────────
# BUG FIX 1: Thêm worker splitting — không có thì 4 workers đọc cùng 4 file
#            → mỗi epoch bị lặp dữ liệu 4 lần.
# BUG FIX 2: persistent_workers=False — với IterableDataset, persistent workers
#            tái sử dụng state cũ, file không được shuffle lại giữa các epoch.
# BUG FIX 3: Thêm legal_mask — không có mask thì model bị phạt trên 4672 action
#            thay vì chỉ các nước hợp lệ, làm chậm hội tụ policy head.
# BUG FIX 4: Guard move idx trong legal_mask (tránh -inf loss spike).
import os, random, collections
import chess, chess.pgn
import numpy as np
import torch
from torch.utils.data import IterableDataset, DataLoader

class _CastlingFixReader:
    def __init__(self, f): self._f = f
    def readline(self):
        return self._f.readline().replace('0-0-0','O-O-O').replace('0-0','O-O')
    def __iter__(self): return self
    def __next__(self):
        l = self.readline()
        if not l: raise StopIteration
        return l

class PGNIterableDataset(IterableDataset):
    def __init__(self, pgn_files, cfg):
        self.pgn_files = list(pgn_files)
        self.cfg       = cfg
        files = list(self.pgn_files)
        random.shuffle(files)
    def __iter__(self):
        # FIX: chia file đều cho từng worker
        worker_info = torch.utils.data.get_worker_info()
        files = list(self.pgn_files)
        if worker_info is not None:
            files = files[worker_info.id :: worker_info.num_workers]

        for pgn_path in files:
            yield from self._parse_file(pgn_path)

    def _parse_file(self, pgn_path):
        try:
            with open(pgn_path, 'r', encoding='utf-8', errors='ignore') as raw:
                rdr = _CastlingFixReader(raw)
                while True:
                    try:    game = chess.pgn.read_game(rdr)
                    except: break
                    if game is None: break
                    if not passes_pgn_filter(game, self.cfg): continue

                    result  = game.headers.get('Result', '*')
                    board   = game.board()
                    history = collections.deque(maxlen=8)

                    for move in game.mainline_moves():
                        history.appendleft(board.copy())
                        boards_list = list(history)
                        turn = board.turn
                        flip = (turn == chess.BLACK)

                        idx = move_to_idx(move, flip=flip)
                        if idx is None:
                            board.push(move); continue

                        state = board_history_to_planes(boards_list, turn)

                        # FIX: build legal mask
                        legal_mask = np.zeros(self.cfg['action_size'], dtype=np.float32)
                        for lm in board.legal_moves:
                            lm_idx = move_to_idx(lm, flip=flip)
                            if lm_idx is not None:
                                legal_mask[lm_idx] = 1.0

                        # FIX: guard idx nằm trong legal mask
                        if legal_mask[idx] == 0:
                            board.push(move); continue

                        policy      = np.zeros(self.cfg['action_size'], dtype=np.float32)
                        policy[idx] = 1.0
                        value       = np.float32(build_value_label(result, turn))
                        board.push(move)

                        yield (
                            torch.from_numpy(state),
                            torch.from_numpy(policy),
                            torch.tensor(value),
                            torch.from_numpy(legal_mask),  # FIX: thêm legal_mask
                        )
        except Exception:
            return

def make_dataloader(pgn_files, cfg, training=True):
    _mb   = sum(os.path.getsize(f) for f in pgn_files) / 1e6
    label = 'train' if training else 'val'
    print(f'   [{label}] {len(pgn_files)} files ({_mb:.0f} MB)')
    ds = PGNIterableDataset(pgn_files, cfg)
    nw = cfg.get('num_workers', 4)
    return DataLoader(
        ds,
        batch_size         = cfg['batch_size'],
        num_workers        = nw,
        pin_memory         = True,
        drop_last          = True,
        prefetch_factor    = 2 if nw > 0 else None,
        persistent_workers = False,  # FIX: False để reshuffle đúng mỗi epoch
    )

print('✅ Dataset ready')


✅ Dataset ready


In [44]:
# ── Cell 9 — ChessTransformer ────────────────────────────────────────────
# BUG FIX: Thêm enable_nested_tensor=False để tắt UserWarning.
# norm_first=True không tương thích với nested tensor optimization của PyTorch.
# Flag này không ảnh hưởng hiệu suất — PyTorch tự fallback về regular tensor
# dù có hay không có cảnh báo — nhưng tắt đi giúp log sạch.
import torch.nn as nn
import torch.nn.functional as F

class ChessTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        in_ch    = cfg['input_planes']
        d_model  = cfg['d_model']
        n_heads  = cfg['n_heads']
        n_layers = cfg['n_layers']
        ffn_dim  = cfg['ffn_dim']
        dropout  = cfg['dropout']
        act_sz   = cfg['action_size']

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, d_model, kernel_size=1, bias=False),
            nn.LayerNorm([d_model, 8, 8]),
        )

        self.pos_emb = nn.Parameter(torch.zeros(1, 64, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = n_heads,
            dim_feedforward = ffn_dim,
            dropout         = dropout,
            activation      = 'gelu',
            batch_first     = True,
            norm_first      = True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers           = n_layers,
            norm                 = nn.LayerNorm(d_model),
            enable_nested_tensor = False,  # FIX: tắt warning, norm_first=True không hỗ trợ
        )

        self.pol_norm = nn.LayerNorm(d_model)
        self.pol_fc   = nn.Linear(64 * d_model, act_sz)

        self.val_norm = nn.LayerNorm(d_model)
        self.val_fc1  = nn.Linear(d_model, 256)
        self.val_fc2  = nn.Linear(256, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.size(0)
        x = self.stem(x)
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_emb
        x = self.transformer(x)

        p = self.pol_norm(x)
        p = self.pol_fc(p.reshape(B, -1))

        v = self.val_norm(x).mean(dim=1)
        v = F.gelu(self.val_fc1(v))
        v = torch.tanh(self.val_fc2(v))

        return p, v

def build_model(cfg):
    model    = ChessTransformer(cfg).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'✅ ChessTransformer — {n_params:,} params')
    print(f'   d_model={cfg["d_model"]} n_heads={cfg["n_heads"]} n_layers={cfg["n_layers"]}')
    return model

print('✅ ChessTransformer ready')


✅ ChessTransformer ready


In [45]:
# ── Cell 10 — Loss & optimizer ────────────────────────────────────────────
import torch.nn as nn
import math

def policy_loss(logits, target, legal_mask=None):
    # FIX: dùng legal_mask để mask nước không hợp lệ trước softmax
    if legal_mask is not None:
        fill_val = torch.finfo(logits.dtype).min
        logits   = logits.masked_fill(legal_mask == 0, fill_val)
    log_prob = torch.nn.functional.log_softmax(logits, dim=-1)
    return -(target * log_prob).sum(dim=-1).mean()

def value_loss(pred, target):
    return torch.nn.functional.mse_loss(pred.squeeze(1), target)

def policy_accuracy(logits, target, legal_mask=None):
    if legal_mask is not None:
        fill_val = torch.finfo(logits.dtype).min
        logits   = logits.masked_fill(legal_mask == 0, fill_val)
    return (logits.argmax(-1) == target.argmax(-1)).float().mean().item()

def policy_top3_accuracy(logits, target, legal_mask=None):
    if legal_mask is not None:
        fill_val = torch.finfo(logits.dtype).min
        logits   = logits.masked_fill(legal_mask == 0, fill_val)
    top3 = logits.topk(3, dim=-1).indices
    true = target.argmax(-1, keepdim=True)
    return (top3 == true).any(-1).float().mean().item()

def build_optimizer(model: nn.Module, cfg: dict):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr           = cfg['learning_rate'],
        weight_decay = cfg['weight_decay'],
        betas        = (0.9, 0.95),
    )
    warmup = cfg.get('warmup_steps', 2_000)
    total  = cfg.get('lr_decay_steps', 200_000)

    def _lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        progress = (step - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

    # FIX: dùng make_grad_scaler() từ Cell 2 — tương thích cả PyTorch 1.x và 2.x
    scaler = make_grad_scaler(enabled=torch.cuda.is_available())

    return optimizer, scheduler, scaler

print('✅ Loss & optimizer ready')


✅ Loss & optimizer ready


In [ ]:
# ── Cell 11 — Training loop ───────────────────────────────────────────────
# BUG FIX: scaler.unscale_(optimizer) bị thiếu trước clip_grad_norm_.
# Khi thiếu, gradient vẫn ở dạng scaled (nhân với ~65536).
# clip_grad_norm_(params, 1.0) sẽ clip ở ngưỡng 65536 thay vì 1.0
# → clipping không bao giờ kích hoạt, gradient spike không bị chặn.
# FIX 2: Guard against empty batches (n==0) to prevent ZeroDivisionError
from tqdm import tqdm
import time

def train_one_epoch(model, loader, optimizer, scheduler, scaler, steps, device):
    model.train()
    total_loss = total_pol = total_val = total_acc = total_top3 = 0.0
    n = 0

    pbar = tqdm(total=steps, desc='  train', leave=False, ncols=80)
    for batch in loader:
        state, pol_target, val_target, legal_mask = batch  # FIX: unpack legal_mask
        state      = state.to(device, non_blocking=True)
        pol_target = pol_target.to(device, non_blocking=True)
        val_target = val_target.to(device, non_blocking=True)
        legal_mask = legal_mask.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # FIX: dùng amp_autocast() từ Cell 2 — tương thích PyTorch 1.x và 2.x
        with amp_autocast(enabled=torch.cuda.is_available()):
            pol_logits, val_pred = model(state)
            p_loss = policy_loss(pol_logits, pol_target, legal_mask)
            v_loss = value_loss(val_pred, val_target)
            loss   = p_loss + v_loss

        scaler.scale(loss).backward()

        # FIX: unscale trước khi clip — bắt buộc để clipping hoạt động đúng
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        total_pol  += p_loss.item()
        total_val  += v_loss.item()
        total_acc  += policy_accuracy(pol_logits.detach(), pol_target, legal_mask)
        total_top3 += policy_top3_accuracy(pol_logits.detach(), pol_target, legal_mask)
        n += 1
        pbar.update(1)
        if n >= steps: break

    pbar.close()
    
    # FIX: guard against n==0 (empty dataloader)
    if n == 0:
        print('   ⚠️  WARNING: No batches processed! Returning zero metrics.')
        return {
            'loss': 0.0, 'pol_loss': 0.0,
            'value_loss': 0.0, 'pol_acc': 0.0, 'pol_top3': 0.0,
        }
    
    return {
        'loss': total_loss/n, 'pol_loss': total_pol/n,
        'value_loss': total_val/n, 'pol_acc': total_acc/n, 'pol_top3': total_top3/n,
    }

@torch.no_grad()
def eval_one_epoch(model, loader, steps, device):
    model.eval()
    total_loss = total_pol = total_val = total_acc = total_top3 = 0.0
    n = 0

    pbar = tqdm(total=steps, desc='  val  ', leave=False, ncols=80)
    for batch in loader:
        state, pol_target, val_target, legal_mask = batch
        state      = state.to(device, non_blocking=True)
        pol_target = pol_target.to(device, non_blocking=True)
        val_target = val_target.to(device, non_blocking=True)
        legal_mask = legal_mask.to(device, non_blocking=True)

        with amp_autocast(enabled=torch.cuda.is_available()):
            pol_logits, val_pred = model(state)
            p_loss = policy_loss(pol_logits, pol_target, legal_mask)
            v_loss = value_loss(val_pred, val_target)

        total_loss += (p_loss + v_loss).item()
        total_pol  += p_loss.item()
        total_val  += v_loss.item()
        total_acc  += policy_accuracy(pol_logits, pol_target, legal_mask)
        total_top3 += policy_top3_accuracy(pol_logits, pol_target, legal_mask)
        n += 1
        pbar.update(1)
        if n >= steps: break

    pbar.close()
    
    # FIX: guard against n==0 (empty dataloader)
    if n == 0:
        print('   ⚠️  WARNING: No validation batches processed! Returning zero metrics.')
        return {
            'loss': 0.0, 'pol_loss': 0.0,
            'value_loss': 0.0, 'pol_acc': 0.0, 'pol_top3': 0.0,
        }
    
    return {
        'loss': total_loss/n, 'pol_loss': total_pol/n,
        'value_loss': total_val/n, 'pol_acc': total_acc/n, 'pol_top3': total_top3/n,
    }

print('✅ Training loop ready')

✅ Training loop ready


In [ ]:
# ── Cell 12 — Full training pipeline ─────────────────────────────────────
import random

def estimate_steps(pgn_files, cfg, sample_games=300):
    files = list(pgn_files); random.shuffle(files)
    n_sampled = n_moves = 0
    for pf in files:
        if n_sampled >= sample_games: break
        try:
            with open(pf, 'r', encoding='utf-8', errors='ignore') as f:
                while n_sampled < sample_games:
                    game = chess.pgn.read_game(f)
                    if game is None: break
                    if passes_pgn_filter(game, cfg):
                        n_moves  += sum(1 for _ in game.mainline_moves())
                        n_sampled += 1
        except: continue
    if n_sampled == 0: 
        print(f'   ⚠️  WARNING: No games passed filter! Check rating_filter={cfg.get("rating_filter", 0)}')
        return 100  # Reduced from 1000 to finish faster if data is bad
    avg   = n_moves / n_sampled
    mb    = sum(os.path.getsize(f) for f in pgn_files) / 1e6
    steps = max(100, int(mb * 1000 / 5 * avg) // cfg['batch_size'])
    print(f'   ~{avg:.0f} moves/game ({n_sampled} games sampled) → {steps:,} steps/epoch')
    return steps

def verify_dataloader(loader, name='Data', min_batches=2):
    """Check if dataloader has at least min_batches available."""
    count = 0
    for _ in loader:
        count += 1
        if count >= min_batches:
            print(f'   ✅ {name} has data (found {min_batches}+ batches)')
            return True
    print(f'   ⚠️  {name} is EMPTY! (got {count} batches)')
    return False

def train_alphazero(cfg, model=None, max_positions=None):
    save_dir = cfg['save_dir']
    device   = DEVICE

    print('\n' + '='*65)
    print('📚 CHESS TRANSFORMER')
    print('='*65)

    print('\n[1/4] Dataloaders...')
    train_loader = make_dataloader(TRAIN_PGN_FILES, cfg, training=True)
    val_loader   = make_dataloader(VAL_PGN_FILES,   cfg, training=False)

    # FIX: Verify dataloaders have data before training
    print('\n[1.5/4] Verifying data...')
    has_train = verify_dataloader(train_loader, 'Train', min_batches=2)
    has_val   = verify_dataloader(val_loader,   'Val',   min_batches=1)
    if not has_train:
        print('\n❌ FATAL: Training set is empty!')
        print('   Suggestions:')
        print(f'   - Reduce rating_filter (currently {cfg.get("rating_filter")})')
        print(f'   - Reduce min_moves (currently {cfg.get("min_moves")})')
        print(f'   - Check PGN file paths: {TRAIN_PGN_FILES[:2]}')
        raise RuntimeError('No training data available')

    print('\n[2/4] Estimating steps...')
    print('  Train:', end=' ')
    train_steps = estimate_steps(TRAIN_PGN_FILES, cfg)
    print('  Val:', end=' ')
    val_steps   = min(1000, max(10, estimate_steps(VAL_PGN_FILES, cfg) // 5))

    if max_positions is not None:
        train_steps = max(1, max_positions // cfg['batch_size'])
        val_steps   = max(1, train_steps // 10)

    setup_tracking(cfg, extra={
        'train_files': len(TRAIN_PGN_FILES),
        'val_files': len(VAL_PGN_FILES),
        'train_steps': train_steps,
        'val_steps': val_steps,
        'max_positions': max_positions,
    })

    print('\n[3/4] Model & optimizer...')
    start_epoch = 1
    net         = build_model(cfg)
    optimizer, scheduler, scaler = build_optimizer(net, cfg)

    if isinstance(model, str):
        print(f'   Loading: {model}')
        ckpt = torch.load(model, map_location=device)
        net.load_state_dict(ckpt.get('model', ckpt))
        if 'optimizer' in ckpt: optimizer.load_state_dict(ckpt['optimizer'])
        if 'scheduler' in ckpt: scheduler.load_state_dict(ckpt['scheduler'])
        if 'scaler'    in ckpt: scaler.load_state_dict(ckpt['scaler'])
        if 'epoch'     in ckpt: start_epoch = ckpt['epoch'] + 1
    elif model is None:
        print('   Training from scratch')

    net = net.to(device)
    if torch.cuda.device_count() > 1:
        net = torch.nn.DataParallel(net)
        print(f'   DataParallel: {torch.cuda.device_count()} GPUs')

    best_acc  = 0.0
    best_path = os.path.join(save_dir, 'checkpoints', 'best.pt')
    history   = []

    print(f'\n[4/4] Training epoch {start_epoch}→{cfg["epochs"]} | '
          f'{train_steps:,} steps/ep | val {val_steps:,} steps')
    print('='*65)

    t0 = time.time()
    for epoch in range(start_epoch, cfg['epochs'] + 1):
        print(f'\nEpoch {epoch}/{cfg["epochs"]}')
        tr = train_one_epoch(net, train_loader, optimizer, scheduler, scaler, train_steps, device)
        vl = eval_one_epoch(net, val_loader, val_steps, device)

        print(f'  pol_acc={tr["pol_acc"]*100:.2f}%  top3={tr["pol_top3"]*100:.2f}%  '
              f'loss={tr["loss"]:.4f}  |  '
              f'val_acc={vl["pol_acc"]*100:.2f}%  val_top3={vl["pol_top3"]*100:.2f}%  '
              f'lr={scheduler.get_last_lr()[0]:.2e}')

        row = {'epoch': epoch,
               **{f'tr_{k}': v for k, v in tr.items()},
               **{f'val_{k}': v for k, v in vl.items()}}
        history.append(row)

        metrics = dict(row)
        metrics['lr'] = scheduler.get_last_lr()[0]
        metrics['elapsed_hours'] = (time.time() - t0) / 3600
        log_tracking(metrics, step=epoch)

        _net = net.module if hasattr(net, 'module') else net
        ckpt = {'epoch': epoch, 'model': _net.state_dict(),
                'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict(), 'cfg': cfg}

        if cfg.get('save_every_epoch'):
            epoch_path = os.path.join(save_dir, 'checkpoints', f'epoch_{epoch:02d}.pt')
            torch.save(ckpt, epoch_path)
            if cfg.get('tracking_upload_checkpoints', False):
                log_tracking_artifact(epoch_path, name='checkpoints')

        if vl['pol_acc'] > best_acc:
            best_acc = vl['pol_acc']
            torch.save(ckpt, best_path)
            if cfg.get('tracking_upload_checkpoints', False):
                log_tracking_artifact(best_path, name='checkpoints')
            print(f'  ✅ Best → {best_acc*100:.2f}%')

    final = os.path.join(save_dir, 'models', 'transformer_final.pt')
    _net  = net.module if hasattr(net, 'module') else net
    torch.save({'model': _net.state_dict(), 'cfg': cfg}, final)
    if cfg.get('tracking_upload_checkpoints', False):
        log_tracking_artifact(final, name='models')

    print(f'\n🏆 DONE  {(time.time()-t0)/3600:.1f}h  best={best_acc*100:.2f}%  → {final}')
    return net, history

print('✅ Pipeline ready')

✅ Pipeline ready


In [50]:
# ── Cell 13 — Start training ──────────────────────────────────────────────

# OPTION A: Full run
trained_model, history = train_alphazero(AZ_CONFIG)

# OPTION B: Debug nhanh (2 epoch, 10k positions)
# trained_model, history = train_alphazero(
#     {**AZ_CONFIG, 'epochs': 2, 'n_layers': 4}, max_positions=10_000
# )

# OPTION C: Resume
# trained_model, history = train_alphazero(
#     AZ_CONFIG,
#     model='/kaggle/working/az_chess_transformer/checkpoints/epoch_03.pt'
# )



📚 CHESS TRANSFORMER

[1/4] Dataloaders...
   [train] 0 files (0 MB)
   [val] 0 files (0 MB)

[2/4] Estimating steps...
  Train:   Val: 
[3/4] Model & optimizer...
✅ ChessTransformer — 86,175,041 params
   d_model=256 n_heads=8 n_layers=12
   Training from scratch
   DataParallel: 2 GPUs

[4/4] Training epoch 1→5 | 1,000 steps/ep | val 200 steps

Epoch 1/5


ZeroDivisionError: float division by zero

In [ ]:
# ── Cell 14 — Plot ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Chess Transformer Training Curves', fontsize=16, fontweight='bold')

plots = [
    ('tr_loss',       'val_loss',       'Total Loss',       axes[0,0]),
    ('tr_pol_loss',   'val_pol_loss',   'Policy Loss',      axes[0,1]),
    ('tr_value_loss', 'val_value_loss', 'Value Loss (MSE)', axes[0,2]),
    ('tr_pol_acc',    'val_pol_acc',    'Policy Top-1 Acc', axes[1,0]),
    ('tr_pol_top3',   'val_pol_top3',   'Policy Top-3 Acc', axes[1,1]),
]

for tr_key, vl_key, title, ax in plots:
    ax.plot(epochs, [h.get(tr_key,0) for h in history], 'b-o', label='Train', markersize=4)
    ax.plot(epochs, [h.get(vl_key,0) for h in history], 'r-o', label='Val',   markersize=4)
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(SAVE_DIR, 'curves.png')
plt.savefig(curves_path, dpi=150)
log_tracking_artifact(curves_path, name='plots')
finish_tracking()
plt.show()


In [ ]:
# ── Cell 15 — Inference demo ──────────────────────────────────────────────
@torch.no_grad()
def predict_moves(model, fen: str, top_k: int = 5):
    board  = chess.Board(fen)
    planes = board_history_to_planes([board], board.turn)
    x      = torch.from_numpy(planes).unsqueeze(0).to(DEVICE)

    _model = model.module if hasattr(model, 'module') else model
    _model.eval()
    pol_logits, value = _model(x)
    pol_logits = pol_logits[0].cpu().numpy()
    value      = float(value[0, 0].cpu())

    flip  = (board.turn == chess.BLACK)
    legal = [(move_to_idx(m, flip), m.uci()) for m in board.legal_moves
             if move_to_idx(m, flip) is not None]

    if not legal: return {'value': value, 'top_moves': []}

    import numpy as np
    mask = np.full(AZ_CONFIG['action_size'], -1e9)
    for i, _ in legal: mask[i] = 0.0
    ml = pol_logits + mask
    ml -= ml[[i for i,_ in legal]].max()
    probs = np.exp(np.array([ml[i] for i,_ in legal]))
    probs /= probs.sum()
    top   = np.argsort(probs)[::-1][:top_k]

    print(f'Value: {value:+.4f}  (from current player POV)')
    for rank, j in enumerate(top, 1):
        bar = '\u2588' * int(probs[j] * 40)
        print(f'  {rank}. {legal[j][1]}  {probs[j]*100:5.1f}%  {bar}')

predict_moves(trained_model, 'r1bq1rk1/pp1pppbp/2n3p1/2P5/8/2P1BNP1/PP1nPP1P/R2QKB1R w KQ - 0 1')
